In [2]:
import pandas as pd
import pyspark
from pyspark.sql import SparkSession


In [60]:
pyspark.__version__
# pyspark.__file__

'4.1.1'

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/08 16:07:45 WARN Utils: Your hostname, codespaces-d816c2, resolves to a loopback address: 127.0.0.1; using 10.0.1.214 instead (on interface eth0)
26/03/08 16:07:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 16:07:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-08 16:08:14--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.238.119, 18.239.238.152, 18.239.238.133, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.238.119|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M   300MB/s    in 0.2s    

2026-03-08 16:08:15 (300 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [5]:
!wc -l yellow_tripdata_2025-11.parquet

282291 yellow_tripdata_2025-11.parquet


In [6]:
df = spark.read \
    .parquet('yellow_tripdata_2025-11.parquet')

In [7]:
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [8]:
df.repartition(4) \
  .write \
  .mode("overwrite") \
  .parquet("pq/")

In [9]:
!ls -lh ./pq


total 102M
-rw-r--r-- 1 codespace codespace   0 Mar  8 16:08 _SUCCESS
-rw-r--r-- 1 codespace codespace 26M Mar  8 16:08 part-00000-393c02c2-7557-4cb5-895c-1d5fd83f0016-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 26M Mar  8 16:08 part-00001-393c02c2-7557-4cb5-895c-1d5fd83f0016-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 26M Mar  8 16:08 part-00002-393c02c2-7557-4cb5-895c-1d5fd83f0016-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 26M Mar  8 16:08 part-00003-393c02c2-7557-4cb5-895c-1d5fd83f0016-c000.snappy.parquet


In [10]:
df.count()

4181444

In [11]:
df.show

<bound method DataFrame.show of DataFrame[VendorID: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: bigint, trip_distance: double, RatecodeID: bigint, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double, cbd_congestion_fee: double]>

In [12]:
df.schema


StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True), StructField('cbd_congestio

In [14]:
df.head(1)

[Row(VendorID=7, tpep_pickup_datetime=datetime.datetime(2025, 11, 1, 0, 13, 25), tpep_dropoff_datetime=datetime.datetime(2025, 11, 1, 0, 13, 25), passenger_count=1, trip_distance=1.68, RatecodeID=1, store_and_fwd_flag='N', PULocationID=43, DOLocationID=186, payment_type=1, fare_amount=14.9, extra=0.0, mta_tax=0.5, tip_amount=1.5, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=22.15, congestion_surcharge=2.5, Airport_fee=0.0, cbd_congestion_fee=0.75)]

In [15]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [19]:
df_pandas_pq = pd.read_parquet("yellow_tripdata_2025-11.parquet")

In [ ]:
# spark.createDataFrame(df_pandas_pq)

In [29]:
from pyspark.sql import functions as F

df.filter(F.to_date(df.tpep_pickup_datetime) == '2025-11-15').head(2)


[Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2025, 11, 15, 0, 0, 16), tpep_dropoff_datetime=datetime.datetime(2025, 11, 15, 0, 7, 22), passenger_count=2, trip_distance=1.27, RatecodeID=1, store_and_fwd_flag='N', PULocationID=137, DOLocationID=113, payment_type=1, fare_amount=8.6, extra=1.0, mta_tax=0.5, tip_amount=2.87, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=17.22, congestion_surcharge=2.5, Airport_fee=0.0, cbd_congestion_fee=0.75),
 Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2025, 11, 15, 0, 3, 1), tpep_dropoff_datetime=datetime.datetime(2025, 11, 15, 0, 29, 53), passenger_count=1, trip_distance=4.25, RatecodeID=1, store_and_fwd_flag='N', PULocationID=249, DOLocationID=237, payment_type=1, fare_amount=26.1, extra=1.0, mta_tax=0.5, tip_amount=3.18, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=35.03, congestion_surcharge=2.5, Airport_fee=0.0, cbd_congestion_fee=0.75)]

In [28]:
df_filter = df.filter(F.to_date(df.tpep_pickup_datetime) == '2025-11-15')
print(df_filter.count())

162604


In [ ]:
F.to_date()

In [39]:
df.registerTempTable('yellow_trip_nov_data')

In [40]:
spark.sql("""
SELECT
    COUNT(*)
FROM
    yellow_trip_nov_data
WHERE
    DATE(tpep_pickup_datetime) = '2025-11-15'
""").show()           

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [42]:
df.withColumn(
    'trip_hours',
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime"))/3600
).select(F.max('trip_hours')).show()

+-----------------+
|  max(trip_hours)|
+-----------------+
|90.64666666666666|
+-----------------+



In [43]:
df_longest_trip = spark.sql("""
SELECT
    MAX(
        (
          unix_timestamp(tpep_dropoff_datetime) -  unix_timestamp(tpep_pickup_datetime)
        ) / 3600
    ) AS longest_trip_hours
FROM
    yellow_trip_nov_data
""").show()

+------------------+
|longest_trip_hours|
+------------------+
| 90.64666666666666|
+------------------+



In [44]:
df_zones = spark.read \
    .option("header", "true") \
    .csv("taxi_zone_lookup.csv")

In [57]:
df_zones.head(3)

[Row(LocationID='1', Borough='EWR', Zone='Newark Airport', service_zone='EWR'),
 Row(LocationID='2', Borough='Queens', Zone='Jamaica Bay', service_zone='Boro Zone'),
 Row(LocationID='3', Borough='Bronx', Zone='Allerton/Pelham Gardens', service_zone='Boro Zone')]

In [45]:
df.registerTempTable('yellow_trip_nov_data')
df_zones.registerTempTable('taxi_zones')

In [58]:
spark.sql("""
SELECT
    tz.Zone,
    COUNT(*) AS trip_count
FROM yellow_trip_nov_data y
JOIN taxi_zones tz
ON y.PULocationID = tz.LocationID
GROUP BY tz.Zone
ORDER BY trip_count ASC
LIMIT 5
""").show(truncate=False)

[Stage 71:=============================>                            (1 + 1) / 2]

+---------------------------------------------+----------+
|Zone                                         |trip_count|
+---------------------------------------------+----------+
|Arden Heights                                |1         |
|Eltingville/Annadale/Prince's Bay            |1         |
|Governor's Island/Ellis Island/Liberty Island|1         |
|Port Richmond                                |3         |
|Rikers Island                                |4         |
+---------------------------------------------+----------+

